# Boltz-2 smoke test — EthnoHack Phase 0

One question: **can we predict binding affinity for one known pair, end to end, on Colab's free tier?**

Pass and the mechanism layer stays in the plan. Fail and we drop it now.

The test pair is carbonic anhydrase II with acetazolamide — binding measured to death in the literature, so a wild prediction is a visible failure rather than a plausible-looking one.

**Before running: Runtime → Change runtime type → T4 GPU.**

The install uses `uv` rather than pip. Plain `pip install boltz` on Colab backtracks through dependency versions and can grind for half an hour; uv resolves in seconds.

Neither the protein sequence nor the ligand structure is typed here — both are fetched, so nothing is transcribed wrongly.

In [ ]:
# 1. Confirm a GPU is attached. Without one, stop here.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Install, with uv and a visible clock.
#    Colab already ships torch, numpy and friends; --system installs into that
#    same environment so they are reused instead of rebuilt.
import time

start = time.time()
!pip install -q uv
!uv pip install --system boltz
print(f'\ninstall took {(time.time() - start) / 60:.1f} minutes')

In [ ]:
# 3. Confirm it imports and record the version for the README.
import boltz

print('boltz', getattr(boltz, '__version__', 'version unknown'))
!boltz --help | head -12

In [ ]:
# 4. Fetch the inputs rather than typing them.
import requests

UNIPROT = 'P00918'   # human carbonic anhydrase 2
LIGAND = 'acetazolamide'

fasta = requests.get(f'https://rest.uniprot.org/uniprotkb/{UNIPROT}.fasta', timeout=60).text
sequence = ''.join(line.strip() for line in fasta.splitlines()[1:])
print(f'{UNIPROT}: {len(sequence)} residues')
print(sequence[:60], '...')

smiles = requests.get(
    f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{LIGAND}/property/CanonicalSMILES/TXT',
    timeout=60,
).text.strip().splitlines()[0]
print(f'\n{LIGAND}: {smiles}')

In [ ]:
# 5. Write the input file Boltz expects.
from pathlib import Path

yaml = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {sequence}
  - ligand:
      id: B
      smiles: '{smiles}'
properties:
  - affinity:
      binder: B
"""

Path('smoke').mkdir(exist_ok=True)
Path('smoke/test_pair.yaml').write_text(yaml)
print(yaml[:260], '...')

In [ ]:
# 6. Predict. Model weights download on this first run (a few GB).
#    The MSA step runs on a remote server and is the slow part — it looks
#    frozen for several minutes. Do not interrupt it.
import time

start = time.time()
!boltz predict smoke/test_pair.yaml --use_msa_server --out_dir smoke_out --override
print(f'\nprediction took {(time.time() - start) / 60:.1f} minutes')

In [ ]:
# 7. Read the answer.
import glob
import json

files = glob.glob('smoke_out/**/affinity*.json', recursive=True)
if not files:
    print('No affinity output. Files produced:')
    for f in sorted(glob.glob('smoke_out/**/*', recursive=True))[:40]:
        print(' ', f)
else:
    data = json.load(open(files[0]))
    print(json.dumps(data, indent=2))
    value = data.get('affinity_pred_value')
    prob = data.get('affinity_probability_binary')
    print('\n--- reading ---')
    if prob is not None:
        print(f'probability it binds at all: {prob:.2f}')
    if value is not None:
        print(f'predicted log10(IC50 in uM): {value:.2f}  ->  IC50 ~ {10**value:.3g} uM')
    print('\nAcetazolamide binds carbonic anhydrase II in the nanomolar range,')
    print('so a sensible prediction is strongly negative on the log10 scale.')
    print('High binding probability plus a low IC50 means PASS.')

## Reading the outcome

**Pass** — probability near 1, small IC50. Saturday's job is then batching 20–50 real pairs.

**Install still slow** — if cell 2 passes about 10 minutes even with uv, stop. The mechanism layer is not worth more of the week.

**Out of memory** — the T4's 16GB could not hold a 260-residue protein. Worth one retry on a smaller target before dropping it.

**Disconnected mid-run** — Colab's timeout. Tells us the real run must write each result to Drive as it completes, holding no state in the notebook.

**Nonsense number** — a known nanomolar binder predicted as weak means the model is not usable for untested compounds either. A finding worth reporting, not a bug to chase.

Record the outcome and the two timings either way. They decide whether the predicted tier exists, and the timings size Saturday's run.